# Deterministic Rules — Full-Population Results EDA

Run on the **AllianceChicago VM** (PHI). We scored every candidate pair with the
deterministic rule cascade (`deterministic_rules/predict_alliance_rules_full.py`) and
got `deterministic_rules_predictions_full_v1.csv` with columns
`PATID_A, PATID_B, rule_pred, rule_id, rule_reason`.

**There is no ground truth on the real population.** So we judge quality indirectly:

1. **Proportions** — how much of the population each decision covers.
2. **Which rules fire** — a healthy run is dominated by interpretable rules, not one catch-all.
3. **Internal consistency** — do the records *behind* each decision actually look like what the
   rule claims? (matches should agree on identifiers; non-matches should genuinely differ;
   review should be genuinely ambiguous). We compute field-agreement rates per decision and
   eyeball real examples.
4. **Merge structure** — treating `match` edges as a graph, are cluster sizes sane or is something over-merging?

Everything here is read-only over the predictions + the cleaned records parquet.

In [ ]:
import os, sys
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
assert os.path.exists('loo.py'), f'Run from the AnyMatch/ repo root (cwd={os.getcwd()!r}).'
sys.path.insert(0, 'deterministic_rules')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils.alliance_schema import id_str_dtypes, prep_record_df, FEATURE_COLS
import rules_alliance as R

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

# ---- paths (edit if your filenames differ) ----
RULES_CSV     = 'data/alliance/deterministic_rules_predictions_full_v1.csv'
RECORDS_PARQUET = 'data/alliance/MDM_Population_cleaned_v4_2026_06_16.parquet'
print('rules :', RULES_CSV)
print('records:', RECORDS_PARQUET)

In [ ]:
# ---- load predictions ----
rules = pd.read_csv(RULES_CSV, dtype={'PATID_A': 'string', 'PATID_B': 'string'})
n = len(rules)
print(f'{n:,} scored pairs')
rules.head()

In [ ]:
# ---- load cleaned records, friendly schema, index by PATID (for joining fields back) ----
records = pd.read_parquet(RECORDS_PARQUET)
for col, dt in id_str_dtypes(records.columns).items():
    records[col] = records[col].astype(dt)
records_valid = records[records['valid_record']].copy()
recs = prep_record_df(records_valid).set_index('PATID')
print(f'{len(records):,} records -> {len(recs):,} valid, indexed by PATID')
print('unique index:', recs.index.is_unique)

## 1. Decision proportions

The headline numbers: of all candidate pairs that survived blocking, how many does the
engine auto-decide vs. send to a human? On a real FQHC population most candidate pairs are
non-matches (blocking is permissive), so expect `non_match` to dominate; `review` should be a
small, manageable queue.

In [ ]:
order = [R.MATCH, R.NON_MATCH, R.REVIEW]
dist = rules['rule_pred'].value_counts().reindex(order, fill_value=0)
summary = pd.DataFrame({'count': dist, 'pct': (dist / n * 100).round(2)})
print(summary.to_string())

decided = int(dist[R.MATCH] + dist[R.NON_MATCH])
print(f'\nAuto-decided: {decided:,} ({decided/n:.1%})   |   Review queue: {dist[R.REVIEW]:,} ({dist[R.REVIEW]/n:.2%})')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
dist.plot.bar(ax=ax[0], color=['#2ca02c', '#d62728', '#ff7f0e'])
ax[0].set_title('Decision counts'); ax[0].set_ylabel('pairs'); ax[0].tick_params(axis='x', rotation=0)
ax[1].pie(dist, labels=order, autopct='%1.1f%%', colors=['#2ca02c', '#d62728', '#ff7f0e'])
ax[1].set_title('Decision share')
plt.tight_layout(); plt.show()

## 2. Which rules fired?

The `rule_id` distribution is the most diagnostic single view. Each id maps 1:1 to a rule in
`RULES.md`. A healthy run spreads matches across the SSN / last-4 / demographic tiers and
doesn't lean on a single catch-all. Watch for: a huge `R-DEMO-MATCH` share (matches with no
strong ID — higher risk), or a huge `R-DEMO-AMBIG` share (big review queue).

In [ ]:
by_rule = (rules.groupby(['rule_id', 'rule_pred']).size().rename('count').reset_index())
tab = by_rule.pivot_table(index='rule_id', columns='rule_pred', values='count', fill_value=0)
tab = tab.reindex(columns=[c for c in order if c in tab.columns], fill_value=0)
tab['total'] = tab.sum(axis=1)
tab['pct'] = (tab['total'] / n * 100).round(2)
tab = tab.sort_values('total', ascending=False)
print(tab.to_string())

tab['total'].sort_values().plot.barh(figsize=(9, 6))
plt.title('Pairs decided by each rule_id'); plt.xlabel('pairs'); plt.tight_layout(); plt.show()

In [ ]:
# Free-text reason breakdown (finer than rule_id; shows e.g. name/dob sub-states)
rules['rule_reason'].value_counts().head(25)

## 3. Internal consistency — do the records behind each decision back it up?

Without labels, our strongest quality signal is: re-derive the comparators on the actual
records and check that each decision class behaves as designed. A `match` bucket should show
high SSN/name/DOB agreement; `non_match` should show low agreement; `review` should sit in
between (the genuinely-hard middle). A `match` bucket with lots of *disagreeing* fields would
be a red flag of over-merging.

In [ ]:
def build_recs(pair_df):
    """Attach _l/_r friendly fields for a sample of pairs (lookup by PATID)."""
    w = pair_df.copy()
    for side, col in [('l', 'PATID_A'), ('r', 'PATID_B')]:
        s = recs.loc[w[col].values, FEATURE_COLS].reset_index(drop=True).add_suffix(f'_{side}')
        w = pd.concat([w.reset_index(drop=True), s], axis=1)
    return w

def agreement_features(w):
    """Re-run the rule comparators on a wide _l/_r frame -> per-pair agreement labels."""
    out = {'name_level': [], 'dob_level': [], 'ssn_full_eq': [], 'last4_eq': [],
           'addr_level': [], 'phone_overlap': [], 'email_eq': []}
    for row in w.itertuples(index=False):
        rd = row._asdict()
        rl = {f: rd.get(f'{f}_l') for f in FEATURE_COLS}
        rr = {f: rd.get(f'{f}_r') for f in FEATURE_COLS}
        fl, fr = R._full_ssn(rl), R._full_ssn(rr)
        out['name_level'].append(R.name_level(rl, rr))
        out['dob_level'].append(R.dob_level(rl, rr))
        out['ssn_full_eq'].append('equal' if (fl and fr and fl == fr) else ('diff' if (fl and fr) else 'na'))
        l4l, l4r = R._last4(rl), R._last4(rr)
        out['last4_eq'].append('equal' if (l4l and l4r and l4l == l4r) else ('diff' if (l4l and l4r) else 'na'))
        out['addr_level'].append(R.addr_level(rl, rr))
        out['phone_overlap'].append(R.phone_overlap(rl, rr))
        out['email_eq'].append(R.email_equal(rl, rr))
    return pd.DataFrame(out)

# Sample within each decision (full re-derivation over millions of pairs is slow).
SAMPLE_PER_CLASS = 4000
rng = np.random.default_rng(42)
samples = []
for dec in order:
    sub = rules[rules['rule_pred'] == dec]
    if len(sub):
        samples.append(sub.sample(min(SAMPLE_PER_CLASS, len(sub)), random_state=42))
samp = pd.concat(samples, ignore_index=True)
samp_feat = pd.concat([samp.reset_index(drop=True), agreement_features(build_recs(samp))], axis=1)
print(f'Re-derived comparators on {len(samp_feat):,} sampled pairs ({SAMPLE_PER_CLASS}/class).')

In [ ]:
# Agreement profile per decision (the sanity table). Read each block as: of pairs the
# engine called X, what fraction agree on each identifier?
for field in ['name_level', 'dob_level', 'ssn_full_eq', 'last4_eq', 'addr_level', 'phone_overlap', 'email_eq']:
    print(f'\n=== {field} by decision (row %) ===')
    ct = pd.crosstab(samp_feat['rule_pred'], samp_feat[field], normalize='index').reindex(order) * 100
    print(ct.round(1).to_string())

## 4. Eyeball real examples per decision

The decisive qualitative check: print actual record pairs side by side for each decision and
each major rule. Do the `match`es look like the same person? Do the `review`s look genuinely
ambiguous (twins / same-name-DOB / movers), not lazy deferrals?

In [ ]:
SHOW_FIELDS = ['first_name', 'middle_name', 'last_name', 'suffix', 'dob', 'ssn', 'ssn4',
               'sex', 'address', 'city', 'state', 'zip', 'phone', 'email']

def show_pair(pa, pb, rule_pred=None, rule_id=None, reason=None):
    a = recs.loc[pa, SHOW_FIELDS]; b = recs.loc[pb, SHOW_FIELDS]
    cmp = pd.DataFrame({'A': a, 'B': b})
    cmp['same'] = (cmp['A'].astype(str).str.upper().str.strip()
                   == cmp['B'].astype(str).str.upper().str.strip())
    hdr = f'A={pa}  B={pb}'
    if rule_pred: hdr += f'  |  {rule_pred} [{rule_id}] {reason}'
    print(hdr); print(cmp.to_string()); print('-' * 80)

def show_examples(decision, rule_id=None, k=5, seed=0):
    sub = rules[rules['rule_pred'] == decision]
    if rule_id: sub = sub[sub['rule_id'] == rule_id]
    if not len(sub):
        print(f'(no pairs for {decision} / {rule_id})'); return
    for r in sub.sample(min(k, len(sub)), random_state=seed).itertuples():
        show_pair(r.PATID_A, r.PATID_B, r.rule_pred, r.rule_id, r.rule_reason)

In [ ]:
print('############### MATCH examples ###############')
show_examples(R.MATCH, k=6, seed=1)

In [ ]:
# Per-rule match examples: SSN-anchored vs demographic-only (the riskier matches)
for rid in ['R-SSN-MATCH', 'R-L4-MATCH', 'R-DEMO-MATCH']:
    print(f'############### MATCH via {rid} ###############')
    show_examples(R.MATCH, rule_id=rid, k=4, seed=2)

In [ ]:
print('############### REVIEW examples (should look genuinely ambiguous) ###############')
show_examples(R.REVIEW, k=8, seed=3)

In [ ]:
print('############### NON_MATCH examples (should be clearly different people) ###############')
show_examples(R.NON_MATCH, k=6, seed=4)

## 5. Merge structure — are matches forming sane clusters?

Treat every `match` pair as an edge in a graph; connected components are the patient
clusters the rules would merge. A few large components are expected (a patient with many
duplicate records), but a giant component swallowing thousands of records is a classic
over-merge symptom (often a transitive chain through a common identifier). The histogram and
the largest-cluster inspection below catch that.

In [ ]:
import networkx as nx
m = rules[rules['rule_pred'] == R.MATCH]
G = nx.Graph()
G.add_edges_from(zip(m['PATID_A'], m['PATID_B']))
comps = sorted(nx.connected_components(G), key=len, reverse=True)
sizes = pd.Series([len(c) for c in comps])
print(f'{G.number_of_edges():,} match edges -> {len(comps):,} clusters covering {G.number_of_nodes():,} records')
print('\nCluster-size distribution:')
print(sizes.value_counts().sort_index().to_string())
print(f'\nlargest cluster: {sizes.max()} records   mean: {sizes.mean():.2f}')

sizes.plot.hist(bins=range(2, int(sizes.max()) + 2), figsize=(9, 4))
plt.title('Match-cluster sizes'); plt.xlabel('records per cluster'); plt.ylabel('# clusters')
plt.tight_layout(); plt.show()

In [ ]:
# Inspect the largest cluster: do these records really look like one patient?
if comps:
    big = list(comps[0])
    print(f'Largest cluster: {len(big)} records')
    display(recs.loc[big, SHOW_FIELDS])

## 6. Takeaways

Fill in after running:
- Decision split (match / non-match / review %) and whether the review queue is human-manageable.
- Which rules carry the matches — strong-ID-anchored (safe) vs. `R-DEMO-MATCH` (demographic-only, watch precision).
- Agreement-table sanity: matches agree on identifiers, non-matches don't, review sits in the middle.
- Any over-merge in the cluster sizes; inspect the largest component.
- Concrete example pairs that look wrong — candidate rule refinements.